# 🧪 Lab Supplement: Raman Spectroscopy

[**Profs. Mike Murphy, Lauren Woods, Chase Slowey, and Jay Foley** — University of North Carolina at Charlotte](https://chemistry.charlotte.edu/)

---

### 🎯 Objectives
- Demonstrate **symmetry principles** in vibrational spectroscopy  
- Predict **IR** and **Raman-active** vibrational modes for a polyatomic molecule  

---

### 📘 Learning Outcomes
By the end of this lab, you should be able to:

1. Identify the **symmetry** of simple molecules  
2. Differentiate between **Raman** and **Infrared (IR)** active vibrational modes  
3. Use **computational and spectroscopic tools** to identify vibrational modes  
4. Calculate **force constants** using spectral data  

---

> 💡 *Tip:*  
> You can run this notebook interactively in Google Colab using the “Open in Colab” button on your Jupyter Book page.





## ⚗️ Approach 

The **FTIR spectrum** for *trans*-dichloroethylene, together with Eqs. (3) and (4), determines three of the four force constants —  
$ k_{\mathrm{CH}} $, $ k_{\theta} $, and $ k_{\tau} $ — within the approximations applied.  

The remaining force constant, $k_{\mathrm{CC}} $, requires measurement of the **Raman spectrum**, because the carbon–carbon stretch $ \bar{\nu}_{\mathrm{CC}} $ does **not** change the molecular dipole moment and is therefore **absent in the FTIR spectrum**.

You will be asked to enter in values for these four force constants, molecular geometries, atomic masses, and all measured FTIR and Raman transitions in wavenumbers as an initial step of this notebook.

## Intuiting the CC force constant
Rather than measure or calculate  $k_{\mathrm{CC}} $ explicitly, you will utilize this notebook to guess and check values within a range that you will set using your chemical intuition.  You will complete select lines of code to complete an implementation of the ${\bf FG}$ matrix method, which given a complete of force constants, molecular geometry, and atomic masses, enables computation of the Raman and FTIR transition frequencies.  The notebook will also provide a slider that you can use to adjust your guess of $k_{\mathrm{CC}} $.  As you adjust the slider, the notebook should update the ${\bf FG}$ matrix, estimate of the frequencies, and provide an analysis of the closeness of the reesults of the ${\bf FG}$ method to your measured values.  Your goal is to minimize the error by identifying the best possible value of $k_{\mathrm{CC}} $.



### 💻 Computational Setup

Several scientific programs can perform these calculations,  
but here we’ll use a **Jupyter notebook** built in **Python** to calculate elements of the **G matrix**.

1. A notebook link is provided on Canvas — open it in a browser.  
2. Click the **🚀 Rocket** icon (top right) and select **“Open in Colab.”**  
3. Go to **File → Save a copy in Drive**, and rename it to: `your_last_name_RamanLab`
4. Run the **first code block** to import all required **functions and constants**.  
When successful, you’ll see a ✅ **green checkmark** next to the cell.

---

> 💡 *Tip:*  
> Remember that FTIR and Raman spectroscopy are **complementary techniques** —  
> together, they allow you to determine all four force constants for the molecule.




## 🧰 Importing Required Libraries

In this notebook, we’ll use a few standard Python libraries:

- **NumPy** — for numerical calculations and array handling  
- **SciPy** — for scientific utilities and signal processing  
- **Matplotlib** — for plotting and data visualization
- **ipywidgets** and **IPython** for interactive slider features

Run the following cell to import the required modules.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
from IPython.display import display, HTML



## 🧾 Preparation

You’ll need your **data collected from Week 1** —  
you *did* bring your lab notebook, right? 😄  

In the next cell, enter the appropriate values after each colon (`:`) for:

- **Bond lengths** (in Å)  
- **Bond angles** (in radians)  
- **Atomic masses** (in amu)  
- **Force constants**
- **Expected Vibrational Frequencies** (in $cm^{-1}$, ordered from largest to smallest values **NOT** in the same order as the worksheet)

Be sure to record your values carefully —  
these will be used in subsequent calculations of the **G matrix** and **force constants**.


In [ ]:
# STUDENT INPUT SECTION
# Replace placeholder values with experimental or calculated ones as directed in lab handout.

# Student input dictionary template
student_input = {
    # Geometry (distances in Å, angles in radians, mass in AMU)
    "tau": np.pi,                      # 0 for cis, π for trans
    "C-C bondlength": 1.31,
    "C-H bondlength": 1.07,
    "C-C-H bond angle": 2 * np.pi / 3, 
    "C mass": 12.01,
    "H mass": 1.008,
    # Force constants (N/cm or Å² N/cm as noted)
    "C-H force constant": 5.217,        # kCH  (N/cm)
    "C-C–C-H interaction": 0.30,       # kCCCH (N/cm)
    "C-C–bend interaction": 0.05,      # kCCth (Å N/cm)
    "C-H–bend interaction": 0.16,      # kCHatha (Å N/cm)
    "bend-bend coupling": 0.00,        # kthathb (Å² N/cm)
    "bend force constant": 0.902,      # ktheta (Å² N/cm)
    "torsion force constant": 0.1893,   # ktau (Å N/cm)
    
    # Expected vibrational frequencies in cm⁻¹ - 
    # 
    "S1": 3086,
    "S2": 3073,
    "S3": 1577,
    "S4": 1271,
    "S5": 1199,
    "S6": 897
    
}


## 🧠 Understanding the Force Constants

Before constructing the **F matrix**, let’s review the meaning of each force constant.  
Each constant describes how strongly the potential energy changes when certain internal coordinates (bond lengths, angles, or torsions) are displaced.

These constants appear in the potential energy expansion:

$$
V = \tfrac{1}{2} \sum_{ij} F_{ij} \, \Delta q_i \, \Delta q_j
$$

where each $q_i$ represents an **internal coordinate** of the molecule  
(e.g., a bond stretch, bend, or torsional displacement).

---

### 📘 Table of Force Constants

| Symbol | Description | Coordinates Coupled | Typical Units |
|:-------|:-------------|:--------------------|:---------------|
| $k_{CC}$ | C=C stretch force constant | $r_{CC}$ | N·cm⁻¹ |
| $k_{CH}$ | C–H stretch force constant | $r_{CH}$ | N·cm⁻¹ |
| $k_{CCCH}$ | Coupling between C=C stretch and C–H stretch | $r_{CC}$, $r_{CH}$ | N·cm⁻¹ |
| $k_{CC\theta}$ | Coupling between C=C stretch and C–C–H bend | $r_{CC}$, $\theta$ | Å·N·cm⁻¹ |
| $k_{CH\alpha\theta}$ | Coupling between C–H stretch and C–C–H bend | $r_{CH}$, $\theta$ | Å·N·cm⁻¹ |
| $k_{\theta}$ | Mean C–C–H bend stiffness | $\theta_a$, $\theta_b$ | Å²·N·cm⁻¹ |
| $k_{\theta_a\theta_b}$ | Coupling between the two C–C–H bending coordinates | $\theta_a$, $\theta_b$ | Å²·N·cm⁻¹ |
| $k_{\tau}$ | Torsional force constant (H–C–C–H twist) | $\tau$ | Å·N·cm⁻¹ |

---

### 💬 Key Idea
- **Diagonal terms** (like $k_{CC}$ or $k_{CH}$) represent independent motions.  
- **Off-diagonal terms** (like $k_{CCCH}$ or $k_{CC\theta}$) describe *coupling* between motions.  
- These couplings explain why vibrational modes are often *mixed* rather than purely stretching or bending.

> 💡 *Later in the analysis*, these constants will determine how energy is shared among vibrations through the F and G matrices.


## 🧮 Constructing the F Matrix

The **F matrix** (force constant matrix) represents how the potential energy of the molecule changes 
with respect to the **internal coordinates** (bond stretches, bends, torsions, and couplings).  

From the *Numerical Calculation* section of your lab handout, the F matrix elements are given by:

$$
\begin{aligned}
F_{11} &= k_{CC}, & F_{12} &= \sqrt{2}\,k_{CCCH}, & F_{13} &= \sqrt{2}\,k_{CCth}, \\
F_{22} &= k_{CH}, & F_{23} &= k_{CHatha}, & F_{33} &= k_{\theta} + k_{thathb}, \\
F_{44} &= k_{CH}, & F_{45} &= k_{CHatha}, & F_{55} &= k_{\theta} - k_{thathb}, \\
F_{66} &= k_{\tau}
\end{aligned}
$$

All other matrix elements are zero.  
The matrix is **symmetric**, so $ F_{ij} = F_{ji} $.  


## To-Do:  Complete build_F_matrix(kCC, params): function

The function below is partially completed.  You will need to complete the lines that assign elements $F_{11}$, $F_{12}$, and $F_{13}$ according to the formulae abouve.  In particular, you need to complete only the lines

```python

    F11 = #<== complete here!
    F12 = #<== complete here!
    F13 = #<== complete here!
```
using correct python / numpy syntax.

In [ ]:
def build_F_matrix(kCC, params):
    """
    Build the F (force constant) matrix for dichloroethylene.
    
    Parameters:
    -----------
    kCC : float
        C-C double bond force constant (N/cm)
    params : dict
        Dictionary containing other force constants
    
    Returns:
    --------
    F : ndarray (6x6)
        Force constant matrix
    """
    # Extract force constants from params
    kCH = params["C-H force constant"]
    kCCCH = params["C-C–C-H interaction"]
    kCCth = params["C-C–bend interaction"]
    kCHatha = params["C-H–bend interaction"]
    kthathb = params["bend-bend coupling"]
    ktheta = params["bend force constant"]
    ktau = params["torsion force constant"]
    
    # Initialize 6x6 symmetric matrix
    F = np.zeros((6, 6))

    # Define elements F11, F12, F13
    ### These will be uncommented in the student version
    #F11 = #<== complete here!
    #F12 = #<== complete here!
    #F13 = #<== complete here!

    
    # Fill non-zero elements according to the provided formula
    ### These will be uncommented in the student version
    #F[0, 0] = F11
    #F[0, 1] = F12
    #F[0, 2] = F13

    ### These next three lines will be absent in the student version
    F[0, 0] = kCC
    F[0, 1] = np.sqrt(2) * kCCCH
    F[0, 2] = np.sqrt(2) * kCCth
    
    F[1, 1] = kCH
    F[1, 2] = kCHatha
    
    F[2, 2] = ktheta + kthathb
    
    F[3, 3] = kCH
    F[3, 4] = kCHatha
    
    F[4, 4] = ktheta - kthathb
    
    F[5, 5] = ktau
    
    # Make symmetric (fill lower triangle)
    F = F + F.T - np.diag(F.diagonal())
    
    return F
    

## ⚙️ Constructing the G Matrix (Full Expressions)

The **G matrix** represents the kinetic coupling between the six internal coordinates
shown in your Figure 2.

From the *Numerical Calculation* section of the lab handout, the matrix elements are:

$$
\begin{aligned}
A &= (\mu_H + \mu_C) \, \rho_{CH}^2, \\
B &= \mu_C \, \rho_{CC} \, (2\rho_{CC} + \rho_{CH}), \\
C &= \mu_C \, \rho_{CC} \left[ \tfrac{\rho_{CC}}{4}(1 + \cos\tau) + \rho_{CH} \right]
\end{aligned}
$$

and the $G$ elements are:

$$
\begin{aligned}
G_{11} &= 2\mu_C, \\
G_{12} &= -\sqrt{\tfrac{1}{2}} \mu_C, \\
G_{13} &= -\sqrt{\tfrac{3}{2}} \rho_{CH}\mu_C, \\
G_{22} &= \mu_C + \mu_H, \\
G_{23} &= -\sqrt{3}\rho_{CC}\mu_C \tfrac{(1 - \cos\tau)}{2}, \\
G_{33} &= A + B(1 - \cos\tau), \\
G_{44} &= G_{22}, \\
G_{45} &= -\sqrt{3}\rho_{CC}\mu_C \tfrac{(1 + \cos\tau)}{2}, \\
G_{55} &= A + B(1 + \cos\tau), \\
G_{66} &= \tfrac{8}{3}\left[A + C(1 + \cos\tau)\right].
\end{aligned}
$$

All other $G_{ij}$ values are zero in this simplified model.

---


## To-Do:  Complete build_G_matrix(params): function

The function below is partially completed.  You will need to complete the lines that assign elements $G_{11}$, $G_{12}$, and $G_{13}$ according to the formulae abouve.  In particular, you need to complete only the lines

```python

    G11 = #<== complete here!
    G12 = #<== complete here!
    G13 = #<== complete here!
```
using correct python / numpy syntax.

> 💡 *Tip:* The diagonal terms represent self-coupling of internal coordinates,  
> while the off-diagonal terms ($G_{12}$, $G_{13}$, etc.) capture kinetic coupling between motions.


In [ ]:
def build_G_matrix(params):
    """
    Build the G (kinematic) matrix for dichloroethylene.
    
    Parameters:
    -----------
    params : dict
        Dictionary containing geometric and mass parameters
    
    Returns:
    --------
    G : ndarray (6x6)
        Kinematic matrix
    """
    # Extract parameters
    tau = params["tau"]
    rho_CC = 1 / params["C-C bondlength"]
    rho_CH = 1 / params["C-H bondlength"]
    m_C = params["C mass"]
    m_H = params["H mass"]
    
    # Convert AMU to appropriate units (1 AMU = 1.66054e-27 kg)
    # For consistency with force constants in N/cm, use reciprocal masses in AMU⁻¹
    mu_C = 1.0 / m_C
    mu_H = 1.0 / m_H
    
    # Calculate intermediate quantities
    A = (mu_H + mu_C) * rho_CH**2
    B = mu_C * rho_CC * (2 * rho_CC + rho_CH)
    C = mu_C * rho_CC * (rho_CC / 4 * (1 + np.cos(tau)) + rho_CH)
    
    # Initialize 6x6 symmetric matrix
    G = np.zeros((6, 6))

    ### These will be uncommented in the student version
    #G11 = #<== complete here!
    #G12 = #<== complete here!
    #G13 = #<== complete here!

    
    
    # Fill non-zero elements according to the provided formula
    ### These will be uncommented in the student version
    #G[0, 0] = G11
    #G[0, 1] = G12
    #G[0, 2] = G13

    G[0, 0] = 2 * mu_C
    G[0, 1] = -np.sqrt(1/2) * mu_C
    G[0, 2] = -np.sqrt(3/2) * rho_CH * mu_C
    
    G[1, 1] = mu_C + mu_H
    G[1, 2] = -np.sqrt(3) * rho_CC * mu_C * (1 - np.cos(tau)) / 2
    
    G[2, 2] = A + B * (1 - np.cos(tau))
    
    G[3, 3] = mu_C + mu_H
    G[3, 4] = -np.sqrt(3) * rho_CC * mu_C * (1 + np.cos(tau)) / 2
    
    G[4, 4] = A + B * (1 + np.cos(tau))
    
    G[5, 5] = (8/3) * (A + C * (1 + np.cos(tau)))
    
    # Make symmetric (fill lower triangle)
    G = G + G.T - np.diag(G.diagonal())
    
    return G


# Putting the functionality together
The next cell will define helper functions that create the interactive capabilities that enable you to adjust a slider to guess-and-check different values of $k_{\rm CC}$.  You need just run these cells, there is nothing to modify here!

In [ ]:
def compute_vibrational_frequencies(kCC, params, verbose=False):
    """
    Compute vibrational frequencies using the Wilson FG method.
    
    Parameters:
    -----------
    kCC : float
        C-C double bond force constant (N/cm)
    params : dict
        Dictionary containing all other parameters
    verbose : bool
        Whether to print detailed output
    
    Returns:
    --------
    frequencies : ndarray
        Vibrational frequencies in cm⁻¹
    """
    # Build F and G matrices
    F = build_F_matrix(kCC, params)
    G = build_G_matrix(params)
    
    if verbose:
        print("F matrix:")
        print(F)
        print("\nG matrix:")
        print(G)
    
    # Compute FG product
    FG = F @ G
    
    if verbose:
        print("\nFG matrix:")
        print(FG)
    
    # Diagonalize FG to get eigenvalues (λ)
    eigenvalues, eigenvectors = np.linalg.eig(FG)
    
    # Sort eigenvalues in descending order
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    
    if verbose:
        print("\nEigenvalues (λ):")
        print(eigenvalues)
    
    # Convert eigenvalues to frequencies
    # ν = (1/2πc) * sqrt(λ) where c is speed of light
    # For our units: ν (cm⁻¹) = 1302.79 * sqrt(λ)
    # The conversion factor accounts for units of F (N/cm) and G (AMU⁻¹)
    conversion_factor = 1302.79  # Converts sqrt(eigenvalue) to cm⁻¹
    frequencies = conversion_factor * np.sqrt(np.abs(eigenvalues))
    
    return frequencies

def compare_with_expected(calculated, params):
    """
    Compare calculated frequencies with expected values.
    
    Parameters:
    -----------
    calculated : ndarray
        Calculated vibrational frequencies
    params : dict
        Dictionary containing expected frequencies
    """
    expected_keys = ["S1", "S2", "S3", "S4", "S5", "S6"]
    expected = [params[key] for key in expected_keys]
    
    print("\n" + "="*70)
    print("COMPARISON WITH EXPECTED VALUES")
    print("="*70)
    print(f"{'Mode':<8} {'Calculated':<15} {'Expected':<15} {'Error':<15} {'% Error':<10}")
    print("-"*70)
    
    total_error = 0
    for i, (calc, exp) in enumerate(zip(calculated, expected)):
        error = calc - exp
        percent_error = (error / exp) * 100 if exp != 0 else 0
        total_error += abs(percent_error)
        print(f"{'S' + str(i+1):<8} {calc:<15.2f} {exp:<15.2f} {error:<15.2f} {percent_error:<10.2f}%")
    
    print("-"*70)
    print(f"Mean Absolute % Error: {total_error/6:.2f}%")
    print("="*70)


def interactive_spectrum(kCC_value):
    """
    Interactive function called by the slider.
    
    Parameters:
    -----------
    kCC_value : float
        C-C double bond force constant from slider
    """
    print(f"\n{'='*70}")
    print(f"C-C FORCE CONSTANT: kCC = {kCC_value:.3f} N/cm")
    print(f"{'='*70}\n")
    
    # Compute frequencies
    frequencies = compute_vibrational_frequencies(kCC_value, student_input, verbose=False)
    
    print("\nCalculated Vibrational Frequencies (cm⁻¹):")
    for i, freq in enumerate(frequencies):
        print(f"  S{i+1}: {freq:.2f} cm⁻¹")
    
    # Compare with expected
    compare_with_expected(frequencies, student_input)
    
    # Create visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar chart comparison
    modes = [f'S{i+1}' for i in range(6)]
    x = np.arange(len(modes))
    width = 0.35
    
    expected = [student_input[f'S{i+1}'] for i in range(6)]
    
    ax1.bar(x - width/2, frequencies, width, label='Calculated', alpha=0.8)
    ax1.bar(x + width/2, expected, width, label='Expected', alpha=0.8)
    ax1.set_xlabel('Vibrational Mode')
    ax1.set_ylabel('Frequency (cm⁻¹)')
    ax1.set_title('Calculated vs Expected Frequencies')
    ax1.set_xticks(x)
    ax1.set_xticklabels(modes)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    
    # Error plot
    errors = [(calc - exp) for calc, exp in zip(frequencies, expected)]
    colors = ['red' if e > 0 else 'blue' for e in errors]
    ax2.bar(modes, errors, color=colors, alpha=0.7)
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
    ax2.set_xlabel('Vibrational Mode')
    ax2.set_ylabel('Error (cm⁻¹)')
    ax2.set_title('Frequency Errors (Calculated - Expected)')
    ax2.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()




# Set your minimum and maximum values for the slider
Below are two variables that will adjust the minimum and maximum values of $k_{\rm CC}$ in $N \cdot m$.  Use your chemical intuition and the existing values of other force constants to decide a reasonable minimum and maximum value to try.


In [ ]:
slider_min = 2 #<== insert your minimum value for k_CC here
slider_max = 10 #<== insert your maximum value for k_CC here

In [ ]:
# Create interactive widget with slider
print("="*70)
print("WILSON FG METHOD: DICHLOROETHYLENE VIBRATIONAL SPECTRUM CALCULATOR")
print("="*70)
print("\nAdjust the C-C force constant slider to optimize the predicted spectrum")
print("Compare your calculated frequencies with the expected values\n")

# Create slider for kCC (typical range for C=C is 9-11 N/cm)
interact(interactive_spectrum, 
         kCC_value=FloatSlider(min=slider_min, max=slider_max, step=0.1, value=slider_min, 
                               description='kCC (N/cm):', 
                               continuous_update=False,
                               style={'description_width': 'initial'}))